# prefixrouter on 2 x T4: does routing by system prompt make the prefix cache pay?

Two vLLM replicas (Qwen2.5-3B-Instruct, one per GPU, prefix caching on) behind the gateway. The load is multi-turn chat: 4 distinct 1,200-token system prompts, 4 sessions each, 4 turns per session. Three conditions, replicas restarted cold before each: round-robin, affinity, and affinity under a skewed mix where one system prompt dominates (fallback valve). ~10 min install, ~6 min per condition.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# vLLM on the Kaggle image: the cu129 release wheel with the matching PyTorch index (the default wheel targets
# CUDA 13), and the image's preinstalled torchaudio/torchcodec removed because they are built for other CUDA
# versions and vLLM imports them (torchcodec is re-pulled by vLLM's requirements, hence the second uninstall).
%pip uninstall -q -y torchaudio torchcodec
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip uninstall -q -y torchcodec
%pip install -q "git+https://github.com/NomanJafar/servellm" "tokenbench[plots] @ git+https://github.com/NomanJafar/tokenbench"
# This repo is private: its wheel arrives as the attached Kaggle dataset prefixrouter-src (see portfolio automation).
!pip install -q /kaggle/input/prefixrouter-src/*.whl
import os; os.makedirs("/kaggle/working/results", exist_ok=True); os.chdir("/kaggle/working")
import torch, vllm, prefixrouter.proxy, servellm.launch, tokenbench
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, torch.cuda.device_count()
import vllm.multimodal.media.audio
print("vllm", vllm.__version__, "torch", torch.__version__, "GPUs", torch.cuda.device_count())

## Replicas, gateway, load

In [ ]:
import asyncio, json, subprocess, time, httpx
from servellm.launch import ServerConfig, VllmServer
from prefixrouter.load import SessionLoad, make_system_prompts, summarize_records

MODEL = "Qwen/Qwen2.5-3B-Instruct"
REPLICAS = ["http://127.0.0.1:8001", "http://127.0.0.1:8002"]
GATEWAY = "http://127.0.0.1:8080"
PROMPTS = make_system_prompts(4, approx_tokens=1200)

def run_sync(coro):
    # Jupyter already runs an event loop, so asyncio.run() raises here; use a worker thread with its own loop.
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(1) as ex:
        return ex.submit(asyncio.run, coro).result()

def start_replicas(tag):
    servers = []
    for gpu, port in ((0, 8001), (1, 8002)):
        # one replica per GPU: the env is captured per server and applied at spawn (run 1 put both on one GPU)
        s = VllmServer(ServerConfig(model=MODEL, dtype="float16", max_model_len=4096, gpu_memory_utilization=0.9, port=port,
                                    extra_args=("--enable-prefix-caching",)), log_path=f"results/startup_{tag}_gpu{gpu}.log",
                       env={"CUDA_VISIBLE_DEVICES": str(gpu)})
        servers.append(s)
    for s in servers: s.start(ready_timeout_s=1200)
    return servers

def start_gateway(mode, threshold):
    p = subprocess.Popen(["prefixrouter", "--replica", REPLICAS[0], "--replica", REPLICAS[1], "--mode", mode,
                          "--load-threshold", str(threshold), "--port", "8080"], stdout=open("results/gateway.log", "a"), stderr=subprocess.STDOUT)
    for _ in range(100):
        try:
            if httpx.get(f"{GATEWAY}/health", timeout=1).status_code == 200: return p
        except httpx.HTTPError: time.sleep(0.3)
    raise RuntimeError("gateway did not start")

def replica_cache_stats():
    out = {}
    for u in REPLICAS:
        t = httpx.get(f"{u}/metrics", timeout=5).text
        q = sum(float(l.split()[-1]) for l in t.splitlines() if l.startswith("vllm:prefix_cache_queries_total"))
        h = sum(float(l.split()[-1]) for l in t.splitlines() if l.startswith("vllm:prefix_cache_hits_total"))
        out[u] = {"queries": q, "hits": h, "hit_rate": (h / q if q else None)}
    return out

CONDITIONS = [
    ("round_robin", "round_robin", 8, PROMPTS, 4),
    ("affinity", "affinity", 8, PROMPTS, 4),
    ("affinity, skewed (one prompt x3)", "affinity", 6, [PROMPTS[0], PROMPTS[0], PROMPTS[0], PROMPTS[1]], 4),
]

In [ ]:
runs = []
for name, mode, threshold, prompts, spp in CONDITIONS:
    tag = name.split(",")[0].replace(" ", "_")
    rec = {"condition": name, "mode": mode, "threshold": threshold, "n_prompts": len(prompts), "sessions_per_prompt": spp}
    servers, gw = [], None
    try:
        servers = start_replicas(tag)
        gw = start_gateway(mode, threshold)
        # warm the engines (CUDA graphs), not the prefix caches: a neutral prompt
        run_sync(SessionLoad(GATEWAY, MODEL, ["warm-up " * 50], sessions_per_prompt=4, turns=1, max_tokens=16).run())
        before = replica_cache_stats()
        t0 = time.time()
        records = run_sync(SessionLoad(GATEWAY, MODEL, prompts, sessions_per_prompt=spp, turns=4, max_tokens=48).run())
        rec["wall_s"] = time.time() - t0
        after = replica_cache_stats()
        rec["summary"] = summarize_records(records)
        rec["cache"] = {u: {"queries": after[u]["queries"] - before[u]["queries"], "hits": after[u]["hits"] - before[u]["hits"]} for u in REPLICAS}
        for u, c in rec["cache"].items(): c["hit_rate"] = c["hits"] / c["queries"] if c["queries"] else None
        rec["gateway_metrics"] = httpx.get(f"{GATEWAY}/metrics").text
        rec["per_request"] = [{"turn": r.meta["turn"], "system_idx": r.meta["system_idx"], "replica": r.meta["replica"], "reason": r.meta["reason"],
                               "ttft_s": r.ttft_s, "prompt_tokens": r.prompt_tokens, "error": r.error} for r in records]
        s = rec["summary"]
        print(f"{name:34s} TTFT p50 {s['ttft']['p50']*1000:.0f} ms p95 {s['ttft']['p95']*1000:.0f} ms | decisions {s['decisions']} | "
              f"hit rates {[round(c['hit_rate'] or 0, 2) for c in rec['cache'].values()]} | split {[v['n'] for v in s['by_replica'].values()]}")
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"; print(name, "FAILED:", rec["error"])
    finally:
        if gw: gw.terminate()
        for s in servers: s.stop()
        time.sleep(5)
    runs.append(rec)
    json.dump(runs, open("results/runs.json", "w"), indent=1)

## Results

In [ ]:
ok = [r for r in runs if "error" not in r]
rows = []
for r in ok:
    s = r["summary"]
    rows.append({"condition": r["condition"], "requests": s["requests"], "errors": s["errors"],
                 "ttft_p50_ms": round(s["ttft"]["p50"] * 1000), "ttft_p95_ms": round(s["ttft"]["p95"] * 1000),
                 **{f"turn{t}_ttft_p50_ms": round(v["ttft"]["p50"] * 1000) for t, v in s["by_turn"].items()},
                 "hit_rate_r1": round(r["cache"][REPLICAS[0]]["hit_rate"] or 0, 2), "hit_rate_r2": round(r["cache"][REPLICAS[1]]["hit_rate"] or 0, 2),
                 "split": "/".join(str(v["n"]) for v in s["by_replica"].values()), "decisions": json.dumps(s["decisions"])})
assert rows, 'every condition failed; see runs.json'
head = list(rows[0].keys())
md_ = "| " + " | ".join(head) + " |\n|" + "---|" * len(head) + "\n" + "\n".join("| " + " | ".join(str(r[h]) for h in head) + " |" for r in rows) + "\n"
open("results/comparison.md", "w").write(md_); json.dump(rows, open("results/comparison.json", "w"), indent=1)
from IPython.display import Markdown, display; display(Markdown(md_))
for r in runs:
    if "error" in r: print("FAILED:", r["condition"], r["error"][:300])

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4.2))
for r in ok:
    bt = r["summary"]["by_turn"]; xs = sorted(bt); ax.plot([x + 1 for x in xs], [bt[x]["ttft"]["p50"] * 1000 for x in xs], marker="o", label=r["condition"])
ax.set_xlabel("conversation turn"); ax.set_ylabel("TTFT p50 (ms)"); ax.set_title("TTFT per turn: round-robin vs prefix affinity (2 x T4, Qwen2.5-3B)"); ax.grid(alpha=.3); ax.legend()
fig.tight_layout(); fig.savefig("results/ttft_by_turn.png", dpi=130)
from IPython.display import Image; display(Image("results/ttft_by_turn.png"))

In [ ]:
!cd results && zip -qr ../results.zip . && ls -la ../results.zip